In [4]:
from typing import TypedDict, List, Union
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

In [2]:
GOOGLE_API_KEY = "***"

In [5]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]

In [6]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY)

In [7]:
def process(state: AgentState) -> AgentState:
    """This node will solve the request you input"""
    response = llm.invoke(state["messages"])

    state["messages"].append(AIMessage(content=response.content)) 
    print(f"\nAI: {response.content}")
    print("CURRENT STATE: ", state["messages"])

    return state

In [8]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END) 
agent = graph.compile()

In [9]:
conversation_history = []

user_input = input("Enter: ")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages": conversation_history})
    conversation_history = result["messages"]
    user_input = input("Enter: ")

Enter:  Hi, I'm Steve M.



AI: Hi Steve! It's great to meet you.

How can I assist you today?
CURRENT STATE:  [HumanMessage(content="Hi, I'm Steve M.", additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Steve! It's great to meet you.\n\nHow can I assist you today?", additional_kwargs={}, response_metadata={})]


Enter:  What's my name?



AI: Your name is Steve M.
CURRENT STATE:  [HumanMessage(content="Hi, I'm Steve M.", additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Steve! It's great to meet you.\n\nHow can I assist you today?", additional_kwargs={}, response_metadata={}), HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Steve M.', additional_kwargs={}, response_metadata={})]


Enter:  exit


In [10]:
with open("logging.txt", "w") as file:
    file.write("Your Conversation Log:\n")
    
    for message in conversation_history:
        if isinstance(message, HumanMessage):
            file.write(f"You: {message.content}\n")
        elif isinstance(message, AIMessage):
            file.write(f"AI: {message.content}\n\n")
    file.write("End of Conversation")

print("Conversation saved to logging.txt")

Conversation saved to logging.txt
